# SBI posterior parameter evaluation

Set `RUN_NAME` in the cell below. Everything else is auto-detected from `run_info.json`.

**Point estimate comparison**: posterior mean vs GT — MAE, Median AE, MAPE  
**Posterior-specific**: 90% credible interval coverage per parameter

In [ ]:
RUN_NAME            = 'exp_cnn4e64-reduced-attnpool_maf5_freeze-maf_1M'
BATCH_IDX           = 0
N_POSTERIOR_SAMPLES = 1000

In [ ]:
import json, sys, importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

ROOT = Path(globals()['_dh'][0])
sys.path.insert(0, str(ROOT))

from dataset import CVDataset, ReducedCVDataset, PARAM_KEYS, PARAM_KEYS_INFER, load_stats

POSTERIOR_PATH = ROOT / f'outputs/{RUN_NAME}/posterior.pt'
STATS_PATH     = ROOT / 'norm_stats.json'
DATA_DIR       = Path('/media/pulsar/SimData/hdf5/cv8/simset_10M_cv8Eed_20260314/test')
MANIFEST       = DATA_DIR.parent / 'manifest_test.json'
BATCH_SIZE     = 256
device         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

run_info = json.load(open(POSTERIOR_PATH.parent / 'run_info.json'))
emb_type = run_info['embedding']['type']

_CONFIG = {
    'WaveformEmbedding':            (CVDataset,        PARAM_KEYS,       'train_sbi', (5, 5)),
    'ReducedWaveformEmbedding':     (ReducedCVDataset, PARAM_KEYS_INFER, 'train_sbi', (6, 4)),
    'TransformerWaveformEmbedding': (ReducedCVDataset, PARAM_KEYS_INFER, 'train_sbi', (6, 4)),
    'ReducedAutoencoderEncoder':    (ReducedCVDataset, PARAM_KEYS_INFER, 'models',    (6, 4)),
}
dataset_cls, PARAM_KEYS_USE, emb_module, (scatter_ncols, scatter_nrows) = _CONFIG[emb_type]

# Inject into __main__ so torch.load can unpickle — posteriors were saved when
# train_sbi.py ran as __main__, so pickle stored the class as __main__.<ClassName>
globals()[emb_type] = getattr(importlib.import_module(emb_module), emb_type)

print(f'Run:       {RUN_NAME}')
print(f'Embedding: {emb_type}')
print(f'Params:    {len(PARAM_KEYS_USE)}  Scatter grid: {scatter_nrows}×{scatter_ncols}')
print(f'Device:    {device}')

## Load posterior and test data

In [ ]:
try:
    posterior = torch.load(POSTERIOR_PATH, map_location=device, weights_only=False)
except (torch.cuda.OutOfMemoryError, RuntimeError):
    print("GPU OOM — falling back to CPU")
    device = torch.device('cpu')
    torch.cuda.empty_cache()
    posterior = torch.load(POSTERIOR_PATH, map_location='cpu', weights_only=False)

print('Posterior loaded:', type(posterior).__name__, f'  device: {device}')

stats    = load_stats(STATS_PATH)
manifest = json.load(open(MANIFEST))

start   = BATCH_IDX * BATCH_SIZE
end     = start + BATCH_SIZE
entries = manifest['index'][start:end]
print(f'Test batch {BATCH_IDX}: sims {start}–{end - 1}  ({len(entries)} entries)')

ds     = dataset_cls(str(DATA_DIR), entries, stats)
loader = DataLoader(ds, batch_size=BATCH_SIZE, num_workers=0)

all_theta, all_x = [], []
for theta_b, x_b in loader:
    all_theta.append(theta_b)
    all_x.append(x_b)
ds.close()

theta_gt = torch.cat(all_theta)
x_test   = torch.cat(all_x)
print(f'GT params: {theta_gt.shape}  observations: {x_test.shape}')

## Embedding diagnostics

In [ ]:
emb_net = posterior.posterior_estimator.embedding_net

with torch.no_grad():
    emb_out = emb_net(x_test[:64].to(device))
print(f'Embedding output shape: {emb_out.shape}')
print(f'Per-dim std  min/mean/max: '
      f'{emb_out.std(0).min():.4f} / {emb_out.std(0).mean():.4f} / {emb_out.std(0).max():.4f}')
print(f'Overall mean: {emb_out.mean():.4f}  std: {emb_out.std():.4f}')

train_data_dir = str(DATA_DIR.parent / 'train')
train_manifest = json.load(open(MANIFEST.parent / 'manifest_train.json'))
ds_tr = dataset_cls(train_data_dir, train_manifest['index'][:1], stats)
theta_tr, x_tr = ds_tr[0]
ds_tr.close()

with torch.no_grad():
    samples_tr = posterior.sample((500,), x=x_tr.to(device), show_progress_bars=False)
post_mean_tr = samples_tr.mean(0).cpu()

print(f"\nTraining obs sanity check:")
print(f"  {'param':<14} {'GT':>10} {'post mean':>12} {'err%':>8}")
for i, name in enumerate(PARAM_KEYS_USE):
    err = abs(post_mean_tr[i].item() - theta_tr[i].item()) / (abs(theta_tr[i].item()) + 1e-9) * 100
    print(f"  {name:<14} {theta_tr[i].item():>10.4f} {post_mean_tr[i].item():>12.4f} {err:>7.1f}%")

## Sample posterior for each test observation

In [ ]:
all_means, all_q05, all_q95 = [], [], []

for i, x_o in enumerate(x_test):
    samples = posterior.sample((N_POSTERIOR_SAMPLES,), x=x_o.to(device), show_progress_bars=False)
    all_means.append(samples.mean(dim=0).cpu())
    all_q05.append(samples.quantile(0.05, dim=0).cpu())
    all_q95.append(samples.quantile(0.95, dim=0).cpu())
    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(x_test)} done')

pred_mean = torch.stack(all_means).numpy()
q05       = torch.stack(all_q05).numpy()
q95       = torch.stack(all_q95).numpy()
gt_phys   = theta_gt.numpy()
print(f'Posterior mean shape: {pred_mean.shape}')

## Point-estimate metrics

In [ ]:
abs_err = np.abs(pred_mean - gt_phys)
ch_mae  = abs_err.mean(axis=0)
ch_med  = np.median(abs_err, axis=0)
ch_mape = (abs_err / (np.abs(gt_phys) + 1e-9) * 100).mean(axis=0)

print(f"{'Parameter':<14} {'MAE':>12} {'Median AE':>12} {'MAPE (%)':>10}")
print('-' * 52)
for i, name in enumerate(PARAM_KEYS_USE):
    print(f'{name:<14} {ch_mae[i]:>12.4f} {ch_med[i]:>12.4f} {ch_mape[i]:>10.2f}%')

print(f'\nOverall  MAE (mean): {ch_mae.mean():.4f}  MAPE: {ch_mape.mean():.2f}%')

In [ ]:
post_mean_std = pred_mean.std(axis=0)
prior_range   = np.array([
    manifest['config']['pvar_high'][k] - manifest['config']['pvar_low'][k]
    for k in PARAM_KEYS_USE
])
variation_pct = post_mean_std / prior_range * 100

print(f"{'param':<14} {'post-mean std':>14} {'prior range':>12} {'variation %':>12}")
print('-' * 55)
for i, name in enumerate(PARAM_KEYS_USE):
    print(f"{name:<14} {post_mean_std[i]:>14.4f} {prior_range[i]:>12.4f} {variation_pct[i]:>11.1f}%")

print(f"\nMean variation across params: {variation_pct.mean():.1f}% of prior range")

In [ ]:
x_pos = np.arange(len(PARAM_KEYS_USE))
fig, axes = plt.subplots(1, 3, figsize=(22, 5))

axes[0].bar(x_pos, ch_mae,  color='steelblue', alpha=0.85)
axes[1].bar(x_pos, ch_med,  color='darkcyan',  alpha=0.85)
axes[2].bar(x_pos, ch_mape, color='tomato',    alpha=0.85)

for ax, title, ylabel in zip(axes,
    ['Mean MAE per parameter', 'Median AE per parameter', 'MAPE per parameter'],
    ['AE (physical units)', 'AE (physical units)', 'MAPE (%)']):
    ax.set_xticks(x_pos)
    ax.set_xticklabels(PARAM_KEYS_USE, rotation=45, ha='right')
    ax.set_title(title)
    ax.set_ylabel(ylabel)

fig.suptitle(f'SBI posterior mean errors — {RUN_NAME}', fontsize=13)
plt.tight_layout()
plt.show()

## Predicted vs GT scatter

In [ ]:
fig, axes = plt.subplots(scatter_nrows, scatter_ncols,
                          figsize=(4 * scatter_ncols, 4 * scatter_nrows))

for i, name in enumerate(PARAM_KEYS_USE):
    ax = axes[i // scatter_ncols, i % scatter_ncols]
    gt_i, pred_i = gt_phys[:, i], pred_mean[:, i]
    lo, hi = gt_i.min(), gt_i.max()
    ax.scatter(gt_i, pred_i, s=10, alpha=0.5, color='steelblue')
    ax.plot([lo, hi], [lo, hi], color='tomato', linewidth=1, linestyle='--')
    ax.set_title(f'{name}  (MAPE {ch_mape[i]:.1f}%)', fontsize=9)
    ax.set_xlabel('GT', fontsize=8)
    ax.set_ylabel('Posterior mean', fontsize=8)
    ax.tick_params(labelsize=7)

for i in range(len(PARAM_KEYS_USE), scatter_nrows * scatter_ncols):
    axes[i // scatter_ncols, i % scatter_ncols].set_visible(False)

fig.suptitle(f'SBI — posterior mean vs GT params  ({RUN_NAME})', fontsize=13)
plt.tight_layout()
plt.show()

## 90% credible interval coverage

For a well-calibrated posterior, ~90% of GT values should fall inside the 90% CI.

In [ ]:
in_ci    = (gt_phys >= q05) & (gt_phys <= q95)
coverage = in_ci.mean(axis=0) * 100

print(f"{'Parameter':<14} {'90% CI coverage':>18}")
print('-' * 35)
for i, name in enumerate(PARAM_KEYS_USE):
    flag = '  ✓' if 85 <= coverage[i] <= 95 else '  ←'
    print(f'{name:<14} {coverage[i]:>16.1f}%{flag}')

print(f'\nMean coverage: {coverage.mean():.1f}%  (target: 90%)')

In [ ]:
x_pos  = np.arange(len(PARAM_KEYS_USE))
fig, ax = plt.subplots(figsize=(14, 4))
colors  = ['steelblue' if 85 <= c <= 95 else 'tomato' for c in coverage]
ax.bar(x_pos, coverage, color=colors, alpha=0.85)
ax.axhline(90, color='black', linestyle='--', linewidth=1, label='target 90%')
ax.axhline(85, color='gray',  linestyle=':',  linewidth=0.8)
ax.axhline(95, color='gray',  linestyle=':',  linewidth=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(PARAM_KEYS_USE, rotation=45, ha='right')
ax.set_ylabel('Coverage (%)')
ax.set_title(f'90% credible interval coverage — {RUN_NAME}')
ax.legend()
plt.tight_layout()
plt.show()